In [ ]:
from google.colab import drive
import os
import sys

drive.mount("/content/drive")

print("Python:", sys.version)
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# We intentionally do NOT install ydata-synthetic.
# Phase 10 will use a TensorFlow/Keras TimeGAN implementation.
print("\nUsing custom TensorFlow/Keras TimeGAN implementation.")

In [ ]:
import pandas as pd
import numpy as np
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

feature_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase5_feature_engineered.parquet"
    )
)

feature_df["Datetime"] = pd.to_datetime(
    feature_df["Datetime"]
)

feature_df = feature_df.sort_values(
    ["StationId", "Datetime"]
).reset_index(drop=True)

print("Dataset:", feature_df.shape)
print("Stations:", feature_df["StationId"].nunique())

In [ ]:
import pandas as pd
import numpy as np
import json

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

PROCESSED_PATH = os.path.join(
    PROJECT_PATH,
    "data",
    "processed"
)

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

feature_df = pd.read_parquet(
    os.path.join(
        PROCESSED_PATH,
        "delhi_phase5_feature_engineered.parquet"
    )
)

feature_df["Datetime"] = pd.to_datetime(
    feature_df["Datetime"]
)

feature_df = feature_df.sort_values(
    ["StationId", "Datetime"]
).reset_index(drop=True)

print("Dataset:", feature_df.shape)
print("Stations:", feature_df["StationId"].nunique())

In [ ]:
unique_times = np.sort(
    feature_df["Datetime"].unique()
)

split_position = int(
    len(unique_times) * 0.80
)

train_end_time = unique_times[
    split_position - 1
]

train_df = feature_df[
    feature_df["Datetime"] <= train_end_time
].copy()

test_df = feature_df[
    feature_df["Datetime"] > train_end_time
].copy()

print("Training:")
print(
    train_df["Datetime"].min(),
    "→",
    train_df["Datetime"].max()
)

print("\nTest:")
print(
    test_df["Datetime"].min(),
    "→",
    test_df["Datetime"].max()
)

In [ ]:
station_counts = (
    train_df
    .groupby(
        ["StationId", "StationName"]
    )
    .size()
    .sort_values(
        ascending=False
    )
)

timegan_station_id = (
    station_counts
    .reset_index()
    .iloc[0]["StationId"]
)

timegan_station_name = (
    station_counts
    .reset_index()
    .iloc[0]["StationName"]
)

print("Station ID:", timegan_station_id)
print("Station:", timegan_station_name)
print("Training records:", station_counts.iloc[0])

In [ ]:
TIMEGAN_FEATURES = [
    "AQI",
    "PM2.5",
    "PM10",
    "NO2",
    "NOx",
    "CO",
    "SO2",
    "O3"
]

SEQ_LEN = 13
HIDDEN_DIM = 32
NOISE_DIM = 32

print("Features:", TIMEGAN_FEATURES)
print("Sequence length:", SEQ_LEN)
print("Hidden dimension:", HIDDEN_DIM)
print("Noise dimension:", NOISE_DIM)

In [ ]:
timegan_station_df = (
    train_df[
        train_df["StationId"] == timegan_station_id
    ]
    .sort_values("Datetime")
    .copy()
)

timegan_station_df = (
    timegan_station_df[
        ["Datetime"] + TIMEGAN_FEATURES
    ]
    .dropna(
        subset=TIMEGAN_FEATURES
    )
    .reset_index(drop=True)
)

print("Usable training rows:", len(timegan_station_df))
display(timegan_station_df.head())

In [ ]:
def build_timegan_sequences(
    df,
    feature_columns,
    sequence_length
):
    values = df[
        feature_columns
    ].to_numpy(
        dtype=np.float32
    )

    timestamps = pd.to_datetime(
        df["Datetime"]
    ).reset_index(drop=True)

    sequences = []

    for i in range(
        sequence_length - 1,
        len(values)
    ):
        window_times = timestamps.iloc[
            i - sequence_length + 1:i + 1
        ]

        if not (
            window_times.diff()
            .dropna()
            .eq(pd.Timedelta(hours=1))
            .all()
        ):
            continue

        sequences.append(
            values[
                i - sequence_length + 1:i + 1
            ]
        )

    return np.asarray(
        sequences,
        dtype=np.float32
    )

real_sequences = build_timegan_sequences(
    timegan_station_df,
    TIMEGAN_FEATURES,
    SEQ_LEN
)

print("Real sequence shape:", real_sequences.shape)

In [ ]:
MAX_TIMEGAN_SEQUENCES = 12000

if len(real_sequences) > MAX_TIMEGAN_SEQUENCES:
    rng = np.random.default_rng(42)

    selected_indices = rng.choice(
        len(real_sequences),
        size=MAX_TIMEGAN_SEQUENCES,
        replace=False
    )

    real_sequences = real_sequences[
        selected_indices
    ]

print(
    "TimeGAN training sequences:",
    len(real_sequences)
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

timegan_scaler = MinMaxScaler(
    feature_range=(-1, 1)
)

real_flat = real_sequences.reshape(
    -1,
    len(TIMEGAN_FEATURES)
)

real_flat_scaled = timegan_scaler.fit_transform(
    real_flat
)

real_sequences_scaled = (
    real_flat_scaled
    .reshape(real_sequences.shape)
    .astype(np.float32)
)

print(
    "Scaled sequence range:",
    real_sequences_scaled.min(),
    "to",
    real_sequences_scaled.max()
)

In [ ]:
import tensorflow as tf
from tensorflow.keras import Model, Sequential
from tensorflow.keras.layers import GRU, Dense

def make_rnn(
    units,
    output_dim,
    output_activation=None
):
    model = Sequential(
        [
            GRU(
                units,
                return_sequences=True
            ),
            GRU(
                units,
                return_sequences=True
            ),
            Dense(
                output_dim,
                activation=output_activation
            )
        ]
    )
    return model


class TimeGAN:
    def __init__(
        self,
        seq_len,
        n_features,
        hidden_dim=32,
        noise_dim=32
    ):
        self.seq_len = seq_len
        self.n_features = n_features
        self.hidden_dim = hidden_dim
        self.noise_dim = noise_dim

        # Embedder: X -> H
        self.embedder = make_rnn(
            hidden_dim,
            hidden_dim,
            "tanh"
        )

        # Recovery: H -> X
        self.recovery = make_rnn(
            hidden_dim,
            n_features,
            "tanh"
        )

        # Generator: Z -> E_hat
        self.generator = make_rnn(
            hidden_dim,
            hidden_dim,
            "tanh"
        )

        # Supervisor: H -> H_hat
        self.supervisor = make_rnn(
            hidden_dim,
            hidden_dim,
            "tanh"
        )

        # Discriminator: H -> probability
        self.discriminator = make_rnn(
            hidden_dim,
            1,
            "sigmoid"
        )

        self.mse = tf.keras.losses.MeanSquaredError()
        self.bce = tf.keras.losses.BinaryCrossentropy()

        self.opt_embedder = tf.keras.optimizers.Adam(
            0.001
        )

        self.opt_supervisor = tf.keras.optimizers.Adam(
            0.001
        )

        self.opt_generator = tf.keras.optimizers.Adam(
            0.001
        )

        self.opt_discriminator = tf.keras.optimizers.Adam(
            0.001
        )

timegan = TimeGAN(
    seq_len=SEQ_LEN,
    n_features=len(TIMEGAN_FEATURES),
    hidden_dim=HIDDEN_DIM,
    noise_dim=NOISE_DIM
)

print("Custom TimeGAN created.")

In [ ]:
@tf.function
def autoencoder_step(
    x
):
    with tf.GradientTape() as tape:
        h = timegan.embedder(x)
        x_hat = timegan.recovery(h)

        reconstruction_loss = tf.reduce_mean(
            tf.square(
                x - x_hat
            )
        )

    variables = (
        timegan.embedder.trainable_variables
        + timegan.recovery.trainable_variables
    )

    gradients = tape.gradient(
        reconstruction_loss,
        variables
    )

    timegan.opt_embedder.apply_gradients(
        zip(
            gradients,
            variables
        )
    )

    return reconstruction_loss


@tf.function
def supervisor_step(
    x
):
    with tf.GradientTape() as tape:

        h = timegan.embedder(x)
        h_supervised = timegan.supervisor(h)

        supervised_loss = tf.reduce_mean(
            tf.square(
                h[:, 1:, :]
                - h_supervised[:, :-1, :]
            )
        )

    gradients = tape.gradient(
        supervised_loss,
        timegan.supervisor.trainable_variables
    )

    timegan.opt_supervisor.apply_gradients(
        zip(
            gradients,
            timegan.supervisor.trainable_variables
        )
    )

    return supervised_loss


@tf.function
def generator_step(
    real_batch
):

    batch_size = tf.shape(
        real_batch
    )[0]

    z = tf.random.normal(
        shape=(
            batch_size,
            timegan.seq_len,
            timegan.noise_dim
        )
    )

    with tf.GradientTape() as tape:

        e_hat = timegan.generator(z)

        h_hat = timegan.supervisor(
            e_hat
        )

        y_fake = timegan.discriminator(
            h_hat
        )

        g_adv_loss = tf.reduce_mean(
            tf.keras.backend.binary_crossentropy(
                tf.ones_like(y_fake),
                y_fake
            )
        )

        # Supervised temporal loss
        real_h = timegan.embedder(
            real_batch
        )

        real_h_next = timegan.supervisor(
            real_h
        )

        g_supervised_loss = tf.reduce_mean(
            tf.square(
                real_h[:, 1:, :]
                - real_h_next[:, :-1, :]
            )
        )

        # Moment matching
        fake_data = timegan.recovery(
            h_hat
        )

        real_mean = tf.reduce_mean(
            real_batch,
            axis=[0, 1]
        )

        fake_mean = tf.reduce_mean(
            fake_data,
            axis=[0, 1]
        )

        real_var = tf.math.reduce_variance(
            real_batch,
            axis=[0, 1]
        )

        fake_var = tf.math.reduce_variance(
            fake_data,
            axis=[0, 1]
        )

        moment_loss = (
            tf.reduce_mean(
                tf.abs(
                    real_mean - fake_mean
                )
            )
            +
            tf.reduce_mean(
                tf.abs(
                    tf.sqrt(real_var + 1e-6)
                    -
                    tf.sqrt(fake_var + 1e-6)
                )
            )
        )

        total_loss = (
            g_adv_loss
            +
            10.0 * tf.sqrt(
                g_supervised_loss + 1e-6
            )
            +
            100.0 * moment_loss
        )

    variables = (
        timegan.generator.trainable_variables
        +
        timegan.supervisor.trainable_variables
    )

    gradients = tape.gradient(
        total_loss,
        variables
    )

    timegan.opt_generator.apply_gradients(
        zip(
            gradients,
            variables
        )
    )

    timegan.opt_supervisor.apply_gradients(
        zip(
            gradients[
                len(
                    timegan.generator.trainable_variables
                ):
            ],
            timegan.supervisor.trainable_variables
        )
    )

    return (
        g_adv_loss,
        g_supervised_loss,
        moment_loss
    )


@tf.function
def discriminator_step(
    real_batch
):

    batch_size = tf.shape(
        real_batch
    )[0]

    z = tf.random.normal(
        shape=(
            batch_size,
            timegan.seq_len,
            timegan.noise_dim
        )
    )

    with tf.GradientTape() as tape:

        real_h = timegan.embedder(
            real_batch
        )

        fake_h = timegan.supervisor(
            timegan.generator(z)
        )

        y_real = timegan.discriminator(
            real_h
        )

        y_fake = timegan.discriminator(
            tf.stop_gradient(fake_h)
        )

        real_loss = tf.reduce_mean(
            tf.keras.backend.binary_crossentropy(
                tf.ones_like(y_real),
                y_real
            )
        )

        fake_loss = tf.reduce_mean(
            tf.keras.backend.binary_crossentropy(
                tf.zeros_like(y_fake),
                y_fake
            )
        )

        total_loss = (
            real_loss
            + fake_loss
        )

    gradients = tape.gradient(
        total_loss,
        timegan.discriminator.trainable_variables
    )

    timegan.opt_discriminator.apply_gradients(
        zip(
            gradients,
            timegan.discriminator.trainable_variables
        )
    )

    return total_loss

In [ ]:
TIMEGAN_BATCH_SIZE = 128

timegan_dataset = (
    tf.data.Dataset
    .from_tensor_slices(
        real_sequences_scaled
    )
    .shuffle(
        min(
            len(real_sequences_scaled),
            10000
        ),
        seed=42
    )
    .batch(
        TIMEGAN_BATCH_SIZE,
        drop_remainder=True
    )
    .prefetch(
        tf.data.AUTOTUNE
    )
)

print("TimeGAN training dataset created.")

In [ ]:
AUTOENCODER_EPOCHS = 100

for epoch in range(
    AUTOENCODER_EPOCHS
):
    epoch_losses = []

    for batch in timegan_dataset:
        loss = autoencoder_step(
            batch
        )

        epoch_losses.append(
            float(loss.numpy())
        )

    if (
        epoch == 0
        or (epoch + 1) % 20 == 0
    ):
        print(
            f"Epoch {epoch + 1}/"
            f"{AUTOENCODER_EPOCHS} "
            f"- Reconstruction loss: "
            f"{np.mean(epoch_losses):.6f}"
        )

print("Embedder/recovery pretraining completed.")

In [ ]:
SUPERVISOR_EPOCHS = 100

for epoch in range(
    SUPERVISOR_EPOCHS
):
    epoch_losses = []

    for batch in timegan_dataset:
        loss = supervisor_step(
            batch
        )

        epoch_losses.append(
            float(loss.numpy())
        )

    if (
        epoch == 0
        or (epoch + 1) % 20 == 0
    ):
        print(
            f"Epoch {epoch + 1}/"
            f"{SUPERVISOR_EPOCHS} "
            f"- Supervisor loss: "
            f"{np.mean(epoch_losses):.6f}"
        )

print("Supervisor pretraining completed.")

In [ ]:
JOINT_EPOCHS = 300

for epoch in range(
    JOINT_EPOCHS
):

    g_adv_losses = []
    g_sup_losses = []
    moment_losses = []
    d_losses = []

    for batch in timegan_dataset:

        g_adv, g_sup, moment = generator_step(
            batch
        )

        d_loss = discriminator_step(
            batch
        )

        g_adv_losses.append(
            float(g_adv.numpy())
        )

        g_sup_losses.append(
            float(g_sup.numpy())
        )

        moment_losses.append(
            float(moment.numpy())
        )

        d_losses.append(
            float(d_loss.numpy())
        )

    if (
        epoch == 0
        or (epoch + 1) % 25 == 0
    ):
        print(
            f"Epoch {epoch + 1}/"
            f"{JOINT_EPOCHS}"
        )

        print(
            "  Generator adversarial:",
            round(
                np.mean(g_adv_losses),
                6
            )
        )

        print(
            "  Generator supervised:",
            round(
                np.mean(g_sup_losses),
                6
            )
        )

        print(
            "  Moment:",
            round(
                np.mean(moment_losses),
                6
            )
        )

        print(
            "  Discriminator:",
            round(
                np.mean(d_losses),
                6
            )
        )

print("TimeGAN joint training completed.")

In [ ]:
def generate_timegan_sequences(
    model,
    n_samples
):
    z = tf.random.normal(
        shape=(
            n_samples,
            model.seq_len,
            model.noise_dim
        )
    )

    generated_latent = (
        model.supervisor(
            model.generator(z),
            training=False
        )
    )

    generated_data = (
        model.recovery(
            generated_latent,
            training=False
        )
    )

    return generated_data.numpy()


N_SYNTHETIC = 3000

synthetic_scaled = generate_timegan_sequences(
    timegan,
    N_SYNTHETIC
)

print(
    "Synthetic scaled sequences:",
    synthetic_scaled.shape
)

In [ ]:
synthetic_flat = synthetic_scaled.reshape(
    -1,
    len(TIMEGAN_FEATURES)
)

synthetic_real_scale = (
    timegan_scaler
    .inverse_transform(
        synthetic_flat
    )
    .reshape(
        synthetic_scaled.shape
    )
    .astype(np.float32)
)

print(
    "Synthetic real-scale shape:",
    synthetic_real_scale.shape
)

In [ ]:
real_flat_for_check = (
    real_sequences
    .reshape(
        -1,
        len(TIMEGAN_FEATURES)
    )
)

synthetic_flat_for_check = (
    synthetic_real_scale
    .reshape(
        -1,
        len(TIMEGAN_FEATURES)
    )
)

quality_rows = []

for i, feature in enumerate(
    TIMEGAN_FEATURES
):

    quality_rows.append(
        {
            "Feature": feature,

            "RealMean": float(
                real_flat_for_check[:, i].mean()
            ),

            "SyntheticMean": float(
                synthetic_flat_for_check[:, i].mean()
            ),

            "RealStd": float(
                real_flat_for_check[:, i].std()
            ),

            "SyntheticStd": float(
                synthetic_flat_for_check[:, i].std()
            )
        }
    )

timegan_quality = pd.DataFrame(
    quality_rows
)

display(timegan_quality)

In [ ]:
# Recreate project output paths
FIGURE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "figures"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

os.makedirs(FIGURE_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)

import matplotlib.pyplot as plt

aqi_feature_index = TIMEGAN_FEATURES.index("AQI")

plt.figure(figsize=(10, 6))

plt.hist(
    real_flat_for_check[:, aqi_feature_index],
    bins=60,
    alpha=0.5,
    label="Real"
)

plt.hist(
    synthetic_flat_for_check[:, aqi_feature_index],
    bins=60,
    alpha=0.5,
    label="Synthetic"
)

plt.title(
    "Real vs Synthetic AQI Distribution"
)

plt.xlabel("AQI")
plt.ylabel("Frequency")
plt.legend()

plt.tight_layout()

plt.savefig(
    os.path.join(
        FIGURE_PATH,
        "phase10_timegan_aqi_distribution.png"
    ),
    dpi=150,
    bbox_inches="tight"
)

plt.show()

print("AQI distribution plot created successfully.")

In [ ]:
real_input = real_sequences_scaled[
    :,
    :12,
    :
]

real_target = (
    real_sequences_scaled[
        :,
        12,
        aqi_feature_index
    ]
)

print(
    "Real input:",
    real_input.shape
)

print(
    "Real target:",
    real_target.shape
)

In [ ]:
synthetic_input_scaled = (
    synthetic_scaled[
        :,
        :12,
        :
    ]
)

synthetic_target_scaled = (
    synthetic_scaled[
        :,
        12,
        aqi_feature_index
    ]
)

print(
    "Synthetic input:",
    synthetic_input_scaled.shape
)

print(
    "Synthetic target:",
    synthetic_target_scaled.shape
)

In [ ]:
SYNTHETIC_RATIO = 0.25

max_synthetic = int(
    len(real_input)
    * SYNTHETIC_RATIO
)

rng = np.random.default_rng(
    42
)

if len(synthetic_input_scaled) > max_synthetic:

    selected = rng.choice(
        len(synthetic_input_scaled),
        size=max_synthetic,
        replace=False
    )

    synthetic_input_used = (
        synthetic_input_scaled[
            selected
        ]
    )

    synthetic_target_used = (
        synthetic_target_scaled[
            selected
        ]
    )

else:

    synthetic_input_used = (
        synthetic_input_scaled
    )

    synthetic_target_used = (
        synthetic_target_scaled
    )

X_augmented = np.concatenate(
    [
        real_input,
        synthetic_input_used
    ],
    axis=0
)

y_augmented = np.concatenate(
    [
        real_target,
        synthetic_target_used
    ],
    axis=0
)

print(
    "Real sequences:",
    len(real_input)
)

print(
    "Synthetic sequences used:",
    len(synthetic_input_used)
)

print(
    "Augmented sequences:",
    len(X_augmented)
)

In [ ]:
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau
)

def create_simple_lstm(
    seq_len,
    n_features
):
    model = tf.keras.Sequential(
        [
            tf.keras.layers.LSTM(
                64,
                input_shape=(
                    seq_len,
                    n_features
                )
            ),

            tf.keras.layers.Dropout(
                0.2
            ),

            tf.keras.layers.Dense(
                32,
                activation="relu"
            ),

            tf.keras.layers.Dense(
                1
            )
        ]
    )

    model.compile(
        optimizer=tf.keras.optimizers.Adam(
            learning_rate=0.001
        ),
        loss="mse",
        metrics=[
            tf.keras.metrics.MeanAbsoluteError(
                name="mae"
            )
        ]
    )

    return model


real_only_lstm = create_simple_lstm(
    12,
    len(TIMEGAN_FEATURES)
)

print("Real-only LSTM created.")

In [ ]:
real_split = int(
    len(real_input) * 0.85
)

X_real_fit = real_input[
    :real_split
]

y_real_fit = real_target[
    :real_split
]

X_real_val = real_input[
    real_split:
]

y_real_val = real_target[
    real_split:
]

print(
    "Real fit:",
    X_real_fit.shape
)

print(
    "Real validation:",
    X_real_val.shape
)

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),

    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

real_history = real_only_lstm.fit(
    X_real_fit,
    y_real_fit,
    validation_data=(
        X_real_val,
        y_real_val
    ),
    epochs=25,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

print("Real-only LSTM trained.")

In [ ]:
augmented_lstm = create_simple_lstm(
    12,
    len(TIMEGAN_FEATURES)
)

print("Augmented LSTM created.")

In [ ]:
# Shuffle only the training examples.
shuffle_indices = np.random.default_rng(
    42
).permutation(
    len(X_augmented)
)

X_augmented = X_augmented[
    shuffle_indices
]

y_augmented = y_augmented[
    shuffle_indices
]

augmented_split = int(
    len(X_augmented) * 0.85
)

X_augmented_fit = X_augmented[
    :augmented_split
]

y_augmented_fit = y_augmented[
    :augmented_split
]

print(
    "Augmented fit:",
    X_augmented_fit.shape
)

print(
    "Validation remains real-only:",
    X_real_val.shape
)

In [ ]:
augmented_history = augmented_lstm.fit(
    X_augmented_fit,
    y_augmented_fit,
    validation_data=(
        X_real_val,
        y_real_val
    ),
    epochs=25,
    batch_size=256,
    callbacks=callbacks,
    verbose=1
)

print("TimeGAN-augmented LSTM trained.")

In [ ]:
station_all = (
    pd.concat(
        [
            train_df[
                train_df["StationId"] == timegan_station_id
            ],
            test_df[
                test_df["StationId"] == timegan_station_id
            ]
        ],
        ignore_index=True
    )
    .sort_values("Datetime")
    .drop_duplicates(
        ["StationId", "Datetime"]
    )
    .dropna(
        subset=TIMEGAN_FEATURES
    )
    .reset_index(drop=True)
)

test_times = set(
    test_df[
        test_df["StationId"] == timegan_station_id
    ]["Datetime"]
)

test_inputs = []
test_targets = []
test_prediction_times = []

values = (
    station_all[
        TIMEGAN_FEATURES
    ]
    .to_numpy(
        dtype=np.float32
    )
)

# Scale using the TimeGAN training scaler only.
values_scaled = (
    timegan_scaler
    .transform(
        values
    )
)

timestamps = (
    station_all["Datetime"]
)

for i in range(
    12,
    len(station_all)
):

    current_time = timestamps.iloc[i]

    if current_time not in test_times:
        continue

    window_times = timestamps.iloc[
        i - 12:i + 1
    ]

    if not (
        window_times.diff()
        .dropna()
        .eq(pd.Timedelta(hours=1))
        .all()
    ):
        continue

    test_inputs.append(
        values_scaled[
            i - 12:i
        ]
    )

    test_targets.append(
        values_scaled[
            i,
            aqi_feature_index
        ]
    )

    test_prediction_times.append(
        current_time
    )

X_timegan_test = np.asarray(
    test_inputs,
    dtype=np.float32
)

y_timegan_test_scaled = np.asarray(
    test_targets,
    dtype=np.float32
)

print(
    "Real test sequences:",
    X_timegan_test.shape
)

In [ ]:
# Recover only AQI using the fitted training scaler.
aqi_scale_min = timegan_scaler.data_min_[
    aqi_feature_index
]

aqi_scale_max = timegan_scaler.data_max_[
    aqi_feature_index
]

def inverse_scale_aqi(
    scaled_values
):
    return (
        (
            scaled_values + 1
        )
        / 2
        * (
            aqi_scale_max
            - aqi_scale_min
        )
        + aqi_scale_min
    )

y_timegan_test = inverse_scale_aqi(
    y_timegan_test_scaled
)

print(
    "Test AQI range:",
    y_timegan_test.min(),
    "→",
    y_timegan_test.max()
)

In [ ]:
real_pred_scaled = (
    real_only_lstm
    .predict(
        X_timegan_test,
        batch_size=256,
        verbose=1
    )
    .ravel()
)

real_pred = inverse_scale_aqi(
    real_pred_scaled
)

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

real_only_mae = mean_absolute_error(
    y_timegan_test,
    real_pred
)

real_only_rmse = np.sqrt(
    mean_squared_error(
        y_timegan_test,
        real_pred
    )
)

real_only_r2 = r2_score(
    y_timegan_test,
    real_pred
)

print("Real-only LSTM")
print("----------------")
print("MAE :", round(real_only_mae, 6))
print("RMSE:", round(real_only_rmse, 6))
print("R²  :", round(real_only_r2, 6))

In [ ]:
augmented_pred_scaled = (
    augmented_lstm
    .predict(
        X_timegan_test,
        batch_size=256,
        verbose=1
    )
    .ravel()
)

augmented_pred = inverse_scale_aqi(
    augmented_pred_scaled
)

augmented_mae = mean_absolute_error(
    y_timegan_test,
    augmented_pred
)

augmented_rmse = np.sqrt(
    mean_squared_error(
        y_timegan_test,
        augmented_pred
    )
)

augmented_r2 = r2_score(
    y_timegan_test,
    augmented_pred
)

print("TimeGAN-Augmented LSTM")
print("----------------------")
print("MAE :", round(augmented_mae, 6))
print("RMSE:", round(augmented_rmse, 6))
print("R²  :", round(augmented_r2, 6))

In [ ]:
timegan_results = pd.DataFrame({
    "Model": [
        "Real-only LSTM",
        "TimeGAN-Augmented LSTM"
    ],
    "MAE": [
        real_only_mae,
        augmented_mae
    ],
    "RMSE": [
        real_only_rmse,
        augmented_rmse
    ],
    "R2": [
        real_only_r2,
        augmented_r2
    ]
})

timegan_results["RMSE_Improvement_%"] = (
    (
        real_only_rmse
        - timegan_results["RMSE"]
    )
    / real_only_rmse
) * 100

display(timegan_results)

In [ ]:
timegan_quality.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase10_timegan_quality.csv"
    ),
    index=False
)

timegan_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase10_timegan_forecasting_results.csv"
    ),
    index=False
)

print("TimeGAN evaluation tables saved.")

In [ ]:
timegan.embedder.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_embedder.keras"
    )
)

timegan.recovery.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_recovery.keras"
    )
)

timegan.generator.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_generator.keras"
    )
)

timegan.supervisor.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_supervisor.keras"
    )
)

timegan.discriminator.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_discriminator.keras"
    )
)

real_only_lstm.save(
    os.path.join(
        MODEL_PATH,
        "phase10_real_only_lstm.keras"
    )
)

augmented_lstm.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_augmented_lstm.keras"
    )

)

print("TimeGAN and forecasting models saved.")

In [ ]:
import joblib

joblib.dump(
    timegan_scaler,
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_scaler.joblib"
    )
)

print("TimeGAN scaler saved.")

In [ ]:
synthetic_sample = synthetic_real_scale[
    :500
]

np.save(
    os.path.join(
        OUTPUT_PATH,
        "phase10_timegan_synthetic_sample.npy"
    ),
    synthetic_sample
)

print(
    "Synthetic sample saved:",
    synthetic_sample.shape
)

In [ ]:
phase10_summary = {
    "completed_phase": "Phase 10 - Custom TimeGAN Augmentation",
    "next_phase": "Phase 11 - Comprehensive Model Comparison and Final Model Selection",

    "station_id": str(
        timegan_station_id
    ),

    "station_name": str(
        timegan_station_name
    ),

    "sequence_length": int(
        SEQ_LEN
    ),

    "features": TIMEGAN_FEATURES,

    "real_sequences": int(
        len(real_sequences)
    ),

    "synthetic_sequences_generated": int(
        len(synthetic_scaled)
    ),

    "synthetic_ratio_used": float(
        SYNTHETIC_RATIO
    ),

    "real_only_lstm": {
        "MAE": float(
            real_only_mae
        ),
        "RMSE": float(
            real_only_rmse
        ),
        "R2": float(
            real_only_r2
        )
    },

    "timegan_augmented_lstm": {
        "MAE": float(
            augmented_mae
        ),
        "RMSE": float(
            augmented_rmse
        ),
        "R2": float(
            augmented_r2
        )
    },

    "rmse_improvement_percent": float(
        (
            real_only_rmse
            - augmented_rmse
        )
        / real_only_rmse
        * 100
    )
}

phase10_summary_path = os.path.join(
    OUTPUT_PATH,
    "phase10_summary.json"
)

with open(
    phase10_summary_path,
    "w"
) as f:
    json.dump(
        phase10_summary,
        f,
        indent=4
    )

print(
    "Phase 10 summary saved:"
)

print(
    phase10_summary_path
)

In [ ]:
print("=" * 60)
print("PHASE 10 — CUSTOM TIMEGAN SUMMARY")
print("=" * 60)

print(
    "Station:",
    timegan_station_name
)

print(
    "Real sequences:",
    len(real_sequences)
)

print(
    "Synthetic sequences:",
    len(synthetic_scaled)
)

print("\nReal-only LSTM:")
print(
    "MAE :",
    round(real_only_mae, 6)
)

print(
    "RMSE:",
    round(real_only_rmse, 6)
)

print(
    "R²  :",
    round(real_only_r2, 6)
)

print("\nTimeGAN-Augmented LSTM:")
print(
    "MAE :",
    round(augmented_mae, 6)
)

print(
    "RMSE:",
    round(augmented_rmse, 6)
)

print(
    "R²  :",
    round(augmented_r2, 6)
)

print(
    "\nRMSE improvement:",
    round(
        (
            real_only_rmse
            - augmented_rmse
        )
        / real_only_rmse
        * 100,
        2
    ),
    "%"
)

print("=" * 60)

In [ ]:
import os
import json
import joblib

PROJECT_PATH = "/content/drive/MyDrive/Delhi-AQI-Prediction"

MODEL_PATH = os.path.join(
    PROJECT_PATH,
    "models"
)

OUTPUT_PATH = os.path.join(
    PROJECT_PATH,
    "outputs"
)

TABLE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "tables"
)

FIGURE_PATH = os.path.join(
    PROJECT_PATH,
    "reports",
    "figures"
)

os.makedirs(MODEL_PATH, exist_ok=True)
os.makedirs(OUTPUT_PATH, exist_ok=True)
os.makedirs(TABLE_PATH, exist_ok=True)
os.makedirs(FIGURE_PATH, exist_ok=True)

# --------------------------------------------------
# 1. Save TimeGAN components
# --------------------------------------------------

timegan.embedder.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_embedder.keras"
    )
)

timegan.recovery.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_recovery.keras"
    )
)

timegan.generator.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_generator.keras"
    )
)

timegan.supervisor.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_supervisor.keras"
    )
)

timegan.discriminator.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_discriminator.keras"
    )
)

# --------------------------------------------------
# 2. Save forecasting models
# --------------------------------------------------

real_only_lstm.save(
    os.path.join(
        MODEL_PATH,
        "phase10_real_only_lstm.keras"
    )
)

augmented_lstm.save(
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_augmented_lstm.keras"
    )
)

# --------------------------------------------------
# 3. Save TimeGAN scaler
# --------------------------------------------------

joblib.dump(
    timegan_scaler,
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_scaler.joblib"
    )
)

# --------------------------------------------------
# 4. Save results
# --------------------------------------------------

timegan_results.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase10_timegan_forecasting_results.csv"
    ),
    index=False
)

timegan_quality.to_csv(
    os.path.join(
        TABLE_PATH,
        "phase10_timegan_quality.csv"
    ),
    index=False
)

# --------------------------------------------------
# 5. Save synthetic sample
# --------------------------------------------------

synthetic_sample = synthetic_real_scale[:500]

np.save(
    os.path.join(
        OUTPUT_PATH,
        "phase10_timegan_synthetic_sample.npy"
    ),
    synthetic_sample
)

# --------------------------------------------------
# 6. Save Phase 10 summary
# --------------------------------------------------

phase10_summary = {
    "completed_phase": "Phase 10 - Custom TimeGAN Augmentation",
    "next_phase": "Phase 11 - Comprehensive Model Comparison and Final Model Selection",

    "station_id": str(timegan_station_id),
    "station_name": str(timegan_station_name),

    "real_sequences": int(len(real_sequences)),
    "synthetic_sequences_generated": int(len(synthetic_scaled)),
    "synthetic_ratio_used": 0.25,

    "real_only_lstm": {
        "MAE": 13.339487,
        "RMSE": 17.651962,
        "R2": 0.975890
    },

    "timegan_augmented_lstm": {
        "MAE": 20.538877,
        "RMSE": 25.042191,
        "R2": 0.951476
    },

    "rmse_improvement_percent": -41.87,

    "conclusion": (
        "TimeGAN augmentation did not improve forecasting "
        "performance in this experiment."
    )
}

summary_path = os.path.join(
    OUTPUT_PATH,
    "phase10_summary.json"
)

with open(
    summary_path,
    "w"
) as f:
    json.dump(
        phase10_summary,
        f,
        indent=4
    )

# --------------------------------------------------
# 7. Save session status
# --------------------------------------------------

session_status = {
    "completed_phase": "Phase 10 - Custom TimeGAN Augmentation",
    "next_phase": "Phase 11 - Comprehensive Model Comparison and Final Model Selection",
    "timegan_improved_model": False
}

status_path = os.path.join(
    OUTPUT_PATH,
    "session_status.json"
)

with open(
    status_path,
    "w"
) as f:
    json.dump(
        session_status,
        f,
        indent=4
    )

# --------------------------------------------------
# 8. Verify checkpoint
# --------------------------------------------------

files_to_check = [
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_embedder.keras"
    ),
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_generator.keras"
    ),
    os.path.join(
        MODEL_PATH,
        "phase10_real_only_lstm.keras"
    ),
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_augmented_lstm.keras"
    ),
    os.path.join(
        MODEL_PATH,
        "phase10_timegan_scaler.joblib"
    ),
    os.path.join(
        TABLE_PATH,
        "phase10_timegan_forecasting_results.csv"
    ),
    os.path.join(
        TABLE_PATH,
        "phase10_timegan_quality.csv"
    ),
    os.path.join(
        OUTPUT_PATH,
        "phase10_timegan_synthetic_sample.npy"
    ),
    summary_path,
    status_path
]

print("=" * 60)
print("PHASE 10 CHECKPOINT SAVED")
print("=" * 60)

for file in files_to_check:
    print(
        "OK   " if os.path.exists(file) else "ERROR",
        file
    )

print("\nCompleted: Phase 10")
print("Next: Phase 11 - Comprehensive Model Comparison and Final Model Selection")